<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/01_nivelacion_ml/13_boosting_stacking.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Boosting y stacking

**Pregunta guía:** ¿Cómo combinamos modelos que corrigen errores distintos?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


## Tres ideas que no deben confundirse

- **Bagging:** modelos en paralelo sobre bootstrap; reduce varianza.
- **Boosting:** modelos secuenciales corrigen residuos o ejemplos
  difíciles. En gradient boosting,
  $F_m(x)=F_{m-1}(x)+\eta h_m(x)$.
- **Stacking:** un metamodelo aprende a combinar predicciones *fuera de
  pliegue* de modelos distintos.

“Blagging” no es un método estándar en este contexto; normalmente se
quiere decir **bagging** o **boosting**.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split

SEMILLA = 42
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)


In [ ]:
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    StackingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(
    n_samples=1_400,
    n_features=16,
    n_informative=7,
    n_redundant=4,
    class_sep=0.9,
    flip_y=0.06,
    random_state=SEMILLA,
)
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEMILLA
)

candidatos = {
    "AdaBoost": (
        AdaBoostClassifier(random_state=SEMILLA),
        {"n_estimators": [50, 150], "learning_rate": [0.03, 0.1, 0.5]},
    ),
    "GradientBoosting": (
        GradientBoostingClassifier(random_state=SEMILLA),
        {
            "n_estimators": [80, 160],
            "learning_rate": [0.03, 0.1],
            "max_depth": [1, 2],
        },
    ),
    "HistGradientBoosting": (
        HistGradientBoostingClassifier(random_state=SEMILLA),
        {"learning_rate": [0.03, 0.1], "max_leaf_nodes": [7, 15, 31]},
    ),
}
filas, búsquedas = [], {}
for nombre, (modelo, grilla) in candidatos.items():
    búsqueda = GridSearchCV(modelo, grilla, scoring="f1", cv=cv, n_jobs=-1).fit(
        X_dev, y_dev
    )
    búsquedas[nombre] = búsqueda
    filas.append(
        {
            "modelo": nombre,
            "F1_CV": búsqueda.best_score_,
            "F1_test": f1_score(y_test, búsqueda.predict(X_test)),
            "mejor": búsqueda.best_params_,
        }
    )
display(pd.DataFrame(filas).set_index("modelo"))


In [ ]:
base = [
    (
        "logística",
        Pipeline(
            [("escala", StandardScaler()), ("modelo", LogisticRegression(max_iter=3000))]
        ),
    ),
    (
        "knn",
        Pipeline(
            [("escala", StandardScaler()), ("modelo", KNeighborsClassifier(15))]
        ),
    ),
    ("árbol", DecisionTreeClassifier(max_depth=5, random_state=SEMILLA)),
]
stacking = StackingClassifier(
    estimators=base,
    final_estimator=LogisticRegression(max_iter=3000),
    cv=cv,
    n_jobs=-1,
)
stacking.fit(X_dev, y_dev)
pred_stack = stacking.predict(X_test)
print(f"F1 test stacking: {f1_score(y_test, pred_stack):.3f}")


En stacking, entrenar el metamodelo con predicciones hechas sobre los
mismos datos usados para ajustar los modelos base produciría fuga. La
implementación genera predicciones fuera de pliegue internamente.

**Ejercicios:** trace error contra número de estimadores; compare tiempos;
cambie el metamodelo; determine si el stacking mejora de forma estable en
diez particiones o sólo en la semilla mostrada.
